Macro Evaluation - Evaluation of the predictions at the global level (mean / average) across images and thresholds

In [1]:
import os, json
from pathlib import Path
from functools import reduce
from collections import defaultdict

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Polygon as MplPolygon
from matplotlib import rcParams
from shapely.geometry import Polygon
import numpy as np
import matplotlib.patheffects as PathEffects
from matplotlib.patches import Patch

import scienceplots
plt.style.use(['science', 'notebook', 'grid', 'no-latex'])

In [2]:
import os 
print(os.getcwd())

c:\Users\abd93000\PycharmProjects\cnt_project_v2\testing\Notebooks\Analysis


In [4]:
input_gt_json_path = '../../../testing/data/annotations_uniques/test/COCO_mask/annotations.json'
output_pred_json_path = '../../../Global_Outputs/edt_comparison/fluo_new_edt/predicted_annotations_poly.json'

outputs_folder = Path('../../../Global_Outputs/edt_comparison')
micro_metric_file = "per_image_threshold_metrics.csv"

In [5]:
with open(input_gt_json_path) as f:
    input_gt_json = json.load(f)

# COCO Evaluation

COCO requires to set parameters of :
* area range
* max number of detections

In [6]:
# Define here the area thresholds for small, medium, and large objects
# Here thresholds are based on 33 and 66 percentiles of bounding box areas

# Compute areas of all bounding boxes
areas = [ann['bbox'][2] * ann['bbox'][3] for ann in input_gt_json['annotations']]

# Calculate thresholds at 33rd and 66th percentiles
q1 = np.percentile(areas, 33)
q2 = np.percentile(areas, 66)

# Define size group function
def size_group(area):
    if area <= q1:
        return 'small'
    elif area <= q2:
        return 'medium'
    else:
        return 'large'

# Count objects in each group
counts = {'small': 0, 'medium': 0, 'large': 0}
for area in areas:
    counts[size_group(area)] += 1

print(f"Thresholds:")
print(f"Small Area<= {q1:.2f}")
print(f"Medium Area> {q1:.2f} and <= {q2:.2f}")
print(f"Large Area> {q2:.2f}")
print("\nNumber of objects in each category:")
for group, count in counts.items():
    print(f"{group.capitalize()}: {count}")

Thresholds:
Small Area<= 120.00
Medium Area> 120.00 and <= 340.00
Large Area> 340.00

Number of objects in each category:
Small: 984
Medium: 939
Large: 983


# Results

Once the area and maxDet are defined

In [7]:
path_output = '../../../Global_Outputs/edt_comparison'

detectron2_list = [
    'detectron2_trial_1'
]

stardist_list = [
    'fluo_new_edt',
]

wormswin_list = [
    'wormswin'
]

nano1d_list = [
    'nano1D'
]

model_names = stardist_list + detectron2_list + wormswin_list + nano1d_list

# SIMPLER NAME TO DISPLAY IN SUMMARY
model_name_map = {
    'Fluo_new_edt': 'NanoVision',
    'wormswin': 'WormSwin',
    'detectron2_trial_1': 'Detectron2',
    'nano1D': 'Nano1D',
}

In [8]:
def load_meso_results(exp_name: str) -> pd.DataFrame:
    """
    Loads the threshold-wise AP results for a given experiment.
    
    Args:
        exp_name (str): The folder name of the experiment inside outputs_folder.
        
    Returns:
        pd.DataFrame: DataFrame with columns renamed to include experiment prefix,
                      except for 'IoU_thresh'.
    """
    if not os.path.exists(os.path.join(path_output, exp_name, "ap_results.csv")):
        raise FileNotFoundError(f"Results file for {exp_name} not found in {path_output}")
    csv_file = os.path.join(path_output, exp_name, "ap_results.csv")
    df = pd.read_csv(csv_file)

    # Prefix all columns except IoU_thresh with the experiment name for clarity
    prefix = exp_name
    df_renamed = df.rename(columns={col: f"{prefix}_{col}" for col in df.columns if col != 'IoU_thresh'})
    
    return df_renamed

# DSB

In [9]:
meso_dfs = [load_meso_results(exp) for exp in model_names]
meso_results = reduce(lambda left, right: pd.merge(left, right, on='IoU_thresh', how='outer'), meso_dfs)

mean_ap = meso_results.drop(columns='IoU_thresh').mean()
print("Mean Average Precision (mAP) per model: \n")
for model_name, ap in mean_ap.items():
    clean_name = model_name.removesuffix('_average_ap')
    mapped_name = model_name_map.get(clean_name, clean_name)
    print(f"{mapped_name}: {ap:.3f}")

Mean Average Precision (mAP) per model: 

fluo_new_edt: 0.269
Detectron2: 0.144
WormSwin: 0.092
Nano1D: 0.021


## COCO

In [10]:
import sys
sys.path.append('../../Evaluation')
# import importlib
# importlib.reload(metrics_utils_coco)
import metrics_utils_coco
from pycocotools import mask as mask_utils

coco_thresholds = np.linspace(0.05, 0.5, int(np.round((0.95 - .5) / .05)) + 1, endpoint=True)

In [11]:
# select some results from coco_eval and display in the dataframe below
results = []

for name in model_names:
    output_pred_json_path = os.path.join(path_output, name, 'predicted_annotations.json')
    coco_eval = metrics_utils_coco.evaluate_coco_annotations(input_gt_json_path, output_pred_json_path, thresholds=coco_thresholds)

    ap_all = coco_eval.stats[0]
    ap_small = coco_eval.stats[3]
    ap_medium = coco_eval.stats[4]
    ap_large = coco_eval.stats[5]

    ar_all_100 = coco_eval.stats[8]
    ar_small = coco_eval.stats[9]
    ar_medium = coco_eval.stats[10]
    ar_large = coco_eval.stats[11]

    print("\n" + "="*40)
    print(f"{name}")
    print(f"AP all (maxDet=100): {ap_all:.3f}, small: {ap_small:.3f}, medium: {ap_medium:.3f}, large: {ap_large:.3f}")
    print(f"AR all (maxDet=100): {ar_all_100:.3f}, small: {ar_small:.3f}, medium: {ar_medium:.3f}, large: {ar_large:.3f}")
    print("="*40)
    print("\n")

    results.append({
        'Experiment': name,
        'AP_all': ap_all,
        'AP_small': ap_small,
        'AP_medium': ap_medium,
        'AP_large': ap_large,
        'AR_all_100': ar_all_100,
        'AR_small': ar_small,
        'AR_medium': ar_medium,
        'AR_large': ar_large
    })

loading annotations into memory...
Done (t=0.05s)
creating index...
index created!
loading annotations into memory...
Done (t=0.05s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *segm*
DONE (t=2.26s).
Accumulating evaluation results...
DONE (t=0.02s).
 Average Precision  (AP) @[ IoU=0.05:0.50 | area=   all | maxDets=100 ] = 0.271
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.042
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.05:0.50 | area= small | maxDets=100 ] = 0.120
 Average Precision  (AP) @[ IoU=0.05:0.50 | area=medium | maxDets=100 ] = 0.349
 Average Precision  (AP) @[ IoU=0.05:0.50 | area= large | maxDets=100 ] = 0.578
 Average Recall     (AR) @[ IoU=0.05:0.50 | area=   all | maxDets= 30 ] = 0.165
 Average Recall     (AR) @[ IoU=0.05:0.50 | area=   all | maxDets= 80 ] = 0.325
 Average Recall     (AR) @[ IoU=0.05:0.50 | area=   all | maxD

TypeError: '<' not supported between instances of 'NoneType' and 'int'

In [14]:
df_results = pd.DataFrame(results).set_index('Experiment')
df_results.index = df_results.index.map(lambda x: model_name_map.get(x, x))
display(df_results)

,AP_all,AP_small,AP_medium,AP_large,AR_all_100,AR_small,AR_medium,AR_large
Experiment,,,,,,,,
fluo_new_edt,0.270737,0.119522,0.348598,0.577992,0.370200,0.248809,0.420725,0.640355
Detectron2,0.186379,0.146535,0.271383,0.212266,0.227632,0.143139,0.276675,0.335025
WormSwin,0.225163,0.110729,0.303613,0.427561,0.293187,0.208956,0.331915,0.444162


mean DICE

In [12]:
# DICE is constant acros all IoU thresholds, so we can use any threshold. Drop the duplicates.
iou_thresh_to_use = 0.5
dice_dfs = []
summary = []

for exp_name in model_names:
    exp_path = outputs_folder / exp_name
    file = exp_path / micro_metric_file

    if not file.exists():
        print(f"Missing file for experiment: {exp_name}")
        continue

    df = pd.read_csv(file)

    if "image_id" not in df.columns or "avg_dice" not in df.columns:
        print(f"Required columns missing in: {exp_name}")
        continue

    # DICE is constant acros all IoU thresholds, so we can use any threshold. Drop the duplicates.
    df_unique_dice = df.drop_duplicates(subset=["image_id"])[["image_id", "avg_dice"]].copy()
    df_unique_dice["experiment"] = exp_name
    dice_dfs.append(df_unique_dice)

    if "IoU_thresh" in df.columns and "TP" in df.columns and "FN" in df.columns:
        metrics_df = df[df["IoU_thresh"] == iou_thresh_to_use].copy()
        metrics_df['total_gt'] = metrics_df['TP'] + metrics_df['FN']
        max_gt = metrics_df['total_gt'].max()
    else:
        print(f"Missing TP/FN/IoU_thresh in: {exp_name}")
        max_gt = None

    avg_dice = df_unique_dice['avg_dice'].mean()
    std_dice = df_unique_dice['avg_dice'].std()
    num_images = df_unique_dice['image_id'].nunique()

    summary.append({
        'Experiment': exp_name,
        'Average Dice': avg_dice,
        'Std Dice': std_dice,
        'Max GT Objects': max_gt,
        'Number of Images': num_images
    })

all_dice_results = pd.concat(dice_dfs, ignore_index=True)
summary_df = pd.DataFrame(summary)



summary_df['Experiment'] = summary_df['Experiment'].map(model_name_map).fillna(summary_df['Experiment'])
summary_df['Average Dice'] = summary_df['Average Dice'].round(2)
summary_df['Max GT Objects'] = summary_df['Max GT Objects'].round(0).astype(int)
print(summary_df.to_string(index=False))

Missing file for experiment: nano1D
  Experiment  Average Dice  Std Dice  Max GT Objects  Number of Images
fluo_new_edt          0.52  0.093891             309                30
  Detectron2          0.41  0.164962             309                30
    WormSwin          0.32  0.209643             309                28
